# 04 · Fine-tune AraBERT (Kaggle)

Fine-tunes [`aubmindlab/bert-base-arabertv02`](https://huggingface.co/aubmindlab/bert-base-arabertv02) for binary sentiment classification on the HARD reviews. Runs on Kaggle for the free GPU (Tesla T4), not locally.

| | |
|---|---|
| **Input** | The train / val / test Parquet files exported by `03_baseline.ipynb`, uploaded as a Kaggle dataset |
| **Output** | Fine-tuned model and tokenizer, pushed to the Hugging Face Hub |

## Check the environment

Confirm a GPU is attached and find where Kaggle mounted the dataset.

In [1]:
import os, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> turn on GPU in Settings!")

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

GPU: Tesla T4
/kaggle/input/datasets/mahmoud5150/hard-sentiment-silver/test.parquet
/kaggle/input/datasets/mahmoud5150/hard-sentiment-silver/train.parquet
/kaggle/input/datasets/mahmoud5150/hard-sentiment-silver/val.parquet


## Load the splits

The dataset path is found with `glob` instead of being hard-coded, since Kaggle's mount path can change. Same splits as the baseline: 82,821 train / 10,217 val / 10,217 test.

In [3]:
import glob, os
from datasets import load_dataset

matches = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
print(matches)

DATA = os.path.dirname(matches[0])
print("using:", DATA)

ds = load_dataset("parquet", data_files={
    "train": f"{DATA}/train.parquet",
    "val":   f"{DATA}/val.parquet",
    "test":  f"{DATA}/test.parquet",
})
print(ds)

['/kaggle/input/datasets/mahmoud5150/hard-sentiment-silver/train.parquet']
using: /kaggle/input/datasets/mahmoud5150/hard-sentiment-silver


DatasetDict({
    train: Dataset({
        features: ['review_id', 'review_clean', 'label'],
        num_rows: 82821
    })
    val: Dataset({
        features: ['review_id', 'review_clean', 'label'],
        num_rows: 10217
    })
    test: Dataset({
        features: ['review_id', 'review_clean', 'label'],
        num_rows: 10217
    })
})


## Tokenize

AraBERT's WordPiece tokenizer splits words into sub-words, e.g. `سيئه` → `سيئ` + `##ه`. So spelling variants of the same word share pieces, unlike the baseline's whole-word vocabulary.

Reviews are truncated to 128 tokens. Padding is left to the data collator, which pads each batch only to its longest review.

In [4]:
from transformers import AutoTokenizer

MODEL = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

print(tokenizer.tokenize("الغرفة سيئه جدا والخدمة ممتازة"))

def tokenize(batch):

    return tokenizer(batch["review_clean"], truncation=True, max_length=128)

ds_tok = ds.map(tokenize, batched=True)

['الغرفة', 'سيئ', '##ه', 'جدا', 'والخدمة', 'ممتازة']


## Model and metrics

The pre-trained AraBERT encoder with a new 2-class classification head on top. The load report is expected: the pre-training heads (`cls.*`) are unused, and the new `classifier` layer starts random and is learned during fine-tuning.

Scored with accuracy and F1, the same metrics as the baseline.

In [5]:
import numpy as np
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

## Train

Standard BERT fine-tuning settings:

- 2 epochs, learning rate 2e-5 with 500 warm-up steps, weight decay 0.01
- batch size 32, mixed precision (`fp16`) to fit the T4 and run faster
- evaluate and save after each epoch, keeping the checkpoint with the best val F1

Training took about 35 minutes.

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["val"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.484009,0.510329,0.905060,0.904802
2,0.399415,0.450233,0.919056,0.919041


TrainOutput(global_step=2590, training_loss=0.5108321819526348, metrics={'train_runtime': 2121.762, 'train_samples_per_second': 78.068, 'train_steps_per_second': 1.221, 'total_flos': 1.051706893818456e+16, 'train_loss': 0.5108321819526348, 'epoch': 2.0})

## Validation result

| Model | Val accuracy | Val F1 |
|---|---|---|
| TF-IDF + logistic regression (`03_baseline`) | 85.15% | 85.15% |
| AraBERT (this notebook) | **91.91%** | **91.90%** |

About 6.8 points better than the baseline, cutting the error rate roughly in half. The model and tokenizer are then saved and zipped for download.

In [8]:
print(trainer.evaluate(ds_tok["val"]))

trainer.save_model("/kaggle/working/arabert-hard")
tokenizer.save_pretrained("/kaggle/working/arabert-hard")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.399415,0.450233,2,0.919056,0.919041


{'eval_loss': 0.45023348927497864, 'eval_accuracy': 0.9190564745032789, 'eval_f1': 0.9190412781991923}


('/kaggle/working/arabert-hard/tokenizer_config.json',
 '/kaggle/working/arabert-hard/tokenizer.json')

In [9]:
import shutil
shutil.make_archive("/kaggle/working/arabert-hard", "zip", "/kaggle/working/arabert-hard")
print("done")

done


## Push to the Hugging Face Hub

The token is entered with `getpass`, so it is never stored in the notebook.

In [16]:
from getpass import getpass
from huggingface_hub import login, logout, whoami

try:
    logout()
except Exception:
    pass

login(token=getpass("HF token: "))

repo = "Mahmoud-103/arabert-hard-sentiment"
print("pushing to:", repo)

model.push_to_hub(repo, private=True)
tokenizer.push_to_hub(repo, private=True)

HF token:  ········


pushing to: Mahmoud-103/arabert-hard-sentiment


CommitInfo(commit_url='https://huggingface.co/Mahmoud-103/arabert-hard-sentiment/commit/da5334b22b2a727f92ec9485a7c5b7a9c684efa7', commit_message='Upload tokenizer', commit_description='', oid='da5334b22b2a727f92ec9485a7c5b7a9c684efa7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Mahmoud-103/arabert-hard-sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='Mahmoud-103/arabert-hard-sentiment'), pr_revision=None, pr_num=None)

## Try it

Load the model back from the Hub and classify two new reviews. `LABEL_1` = positive, `LABEL_0` = negative.

- `الغرفة نظيفة والموظفين متعاونين جدا` ("The room is clean and the staff very helpful") → positive, 97.5%
- `سيء جدا لا انصح به ابدا` ("Very bad, I don't recommend it at all") → negative, 99.9%

In [17]:
from transformers import pipeline

clf = pipeline("text-classification", model="Mahmoud-103/arabert-hard-sentiment")

print(clf("الغرفة نظيفة والموظفين متعاونين جدا"))
print(clf("سيء جدا لا انصح به ابدا"))

[{'label': 'LABEL_1', 'score': 0.9754422307014465}]
[{'label': 'LABEL_0', 'score': 0.9992074370384216}]
